# Learning-rate sweep — RD curves

Rate–distortion curves (PSNR-to-MERLIN vs bitstream bpp on `test_sub500`) for the four
architectures (FP, ResFP, SH, ResSH) across the four swept learning rates
(`1e-5`, `5e-5`, `1e-4`, `5e-4`).

**Two complete curves per arch** (10 λ × 6 seeds = 60 runs): the old `5e-5` default and the
production lr. **Two coarse curves** (3 λ × 2–3 seeds = 6–9 runs) from the exploratory mini-sweep.
Production lr: FP / ResFP / SH → `5e-4`, ResSH → `1e-4`.

Filtering is purely `no_output_padding==True` + dedup on `(arch, lr, seed, λ)` — the complete
(60) and coarse (6–9) curves separate naturally from the run counts, no tag filtering needed.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

ROOT_DIR = Path("..").resolve()
WANDB_CSV = ROOT_DIR / "notebooks" / "SAR_DDC_FPGA_all_runs_WandB.csv"
import sys as _sys

_sys.path.insert(0, str(ROOT_DIR / "notebooks"))
del _sys
from _plotkit import PALETTE

PAL = PALETTE["architectures"]  # Okabe-Ito, colorblind-safe

SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "lr-sweep"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Metrics + axes ──────────────────────────────────────────────────────
P = "test_sub500/psnr_merlin"
B = "test_sub500/bpp_bitstream"

LRS = [1e-5, 5e-5, 1e-4, 5e-4]
LRMK = {1e-5: "v", 5e-5: "o", 1e-4: "^", 5e-4: "s"}  # marker = lr
LRLBL = {1e-5: "1e-5", 5e-5: "5e-5", 1e-4: "1e-4", 5e-4: "5e-4"}
LRCLR = {1e-5: "#999999", 5e-5: "#CC79A7", 1e-4: "#009E73", 5e-4: "#D55E00"}

# (code, label, production/best lr)
ARCHS = [
    ("FP", "FP", 5e-4),
    ("ResFP", "ResFP", 5e-4),
    ("SHyp", "SH", 5e-4),
    ("ResSHyp", "ResSH", 1e-4),
]

COMPLETE_MIN = 30  # >= this many runs ⇒ "full" sweep curve (else coarse)

In [ ]:
# ── Load + prepare ──────────────────────────────────────────────────────
raw = pd.read_csv(WANDB_CSV)
raw["no_output_padding"] = raw["no_output_padding"].map(
    {"True": True, "False": False, True: True, False: False}
)

# ── Quality filters (same set as all other notebooks) ───────────────────
raw["tags"] = raw["tags"].apply(lambda v: json.loads(v) if isinstance(v, str) else [])
raw["seed"] = pd.to_numeric(raw["seed"], errors="coerce")

# Dataset, seeds 0-5, no broken/debug runs, no GDN variants
d = raw[raw["data_name"].isin(["TSXSSCDataModule"])].copy()
d = d[d["seed"].isin(range(6))]
for tag in ("debug", "crashed"):
    d = d[~d["tags"].apply(lambda ts: tag in ts)]
d = d[~d["model_name"].str.contains("gdn1", case=False, na=False)]

# no_output_padding=True isolates the relu production config; dedup by (arch, lr, seed, λ).
d = d[d["no_output_padding"] == True].copy()
d = d.sort_index().drop_duplicates(["architecture", "lr", "seed", "lmbda"], keep="last")

agg = (
    d.groupby(["architecture", "lr", "lmbda"])
    .agg(psnr=(P, "mean"), bpp=(B, "mean"), n=("seed", "size"))
    .reset_index()
)
nrun = d.groupby(["architecture", "lr"]).size()

# ── Coverage: #runs per (arch, lr) ─ guideline: 60 = full, 6/9 = coarse ──
print("matches per (arch, lr)  [n / nλ / nseed]")
for code, lbl, _ in ARCHS:
    print(f"\n{lbl}:")
    for lr in LRS:
        s = d[(d.architecture == code) & (d.lr.round(8) == round(lr, 8))]
        kind = "COMPLETE" if len(s) >= COMPLETE_MIN else "sweep"
        print(
            f"  lr={lr:.0e}: n={len(s):2d}  ({s.lmbda.nunique()}λ × {s.seed.nunique()} seed)  {kind}"
        )

## Per-architecture RD curves (one subplot per arch)

Color = lr, bold solid = full sweep (60 runs), dashed = coarse sweep (6–9 runs). One common
legend below the four subplots.

In [ ]:
def plot_lr_per_arch(agg, nrun, show_coarse=True):
    """4 subplots (FP, ResFP, SH, ResSH); 4 lr curves each. Color=arch; marker=lr; dashed=coarse sweep."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    for ax, (code, lbl, _) in zip(axes.flat, ARCHS):
        a = agg[agg.architecture == code]
        col = PAL[code]
        for lr in LRS:
            complete = nrun.get((code, lr), 0) >= COMPLETE_MIN
            if not complete and not show_coarse:
                continue
            s = a[a.lr.round(8) == round(lr, 8)].sort_values("bpp")
            if s.empty:
                continue
            ax.plot(
                s.bpp,
                s.psnr,
                marker=LRMK[lr],
                color=col,
                ms=6,
                lw=2.8 if complete else 1.5,
                ls="-" if complete else "--",
                alpha=1.0 if complete else 0.8,
                zorder=5 if complete else 3,
            )
        ax.grid(alpha=0.3)
        ax.set_title(lbl, fontweight="bold", fontsize=13)
        ax.set_xlabel("Bitrate [bpp]")
        ax.set_ylabel("PSNR [dB]")
    # Enforce the same PSNR Y-axis range across all subplots
    ymins = [ax.get_ylim()[0] for ax in axes.flat]
    ymaxs = [ax.get_ylim()[1] for ax in axes.flat]
    y_range = (min(ymins), max(ymaxs))
    for ax in axes.flat:
        ax.set_ylim(y_range)
    # Legend: marker shape → learning rate (color is arch, shown by subplot title)
    lr_handles = [
        Line2D([0], [0], color="0.3", marker=LRMK[lr], ls="-", ms=7, label=LRLBL[lr]) for lr in LRS
    ]
    fig.legend(
        lr_handles,
        [LRLBL[lr] for lr in LRS],
        loc="lower center",
        ncol=4,
        fontsize=11,
        frameon=True,
        bbox_to_anchor=(0.5, -0.01),
        title="learning rate  (solid/bold = full sweep, dashed = coarse sweep)",
    )
    fig.suptitle("lr-sweep — RD curves per architecture (test_sub500, MERLIN ref)", fontsize=14)
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


fig = plot_lr_per_arch(agg, nrun, show_coarse=True)
if SAVE_FIGURES:
    fig.savefig(PLOTS_DIR / f"lr_sweep_per_arch.{FIGURE_FORMAT}", bbox_inches="tight")
    print("saved", PLOTS_DIR / f"lr_sweep_per_arch.{FIGURE_FORMAT}")
plt.show()

## Single-axes overlay (all 4 archs)

Color = arch (Okabe-Ito palette), marker = lr, **bold = production lr**. `show_coarse=False`
keeps only the full (60-run) sweeps; `show_coarse=True` adds the dashed exploratory curves
(heavier, diagnostic only).

In [ ]:
def plot_lr_overlay(agg, nrun, show_coarse=False, ax=None):
    """All 4 archs on one axes. Color=arch, marker=lr, bold=production lr.
    show_coarse=False hides the 6-9-run exploratory curves (keeps only the 60-run ones)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 7))
    for code, lbl, best in ARCHS:
        col = PAL[code]
        a = agg[agg.architecture == code]
        for lr in LRS:
            complete = nrun.get((code, lr), 0) >= COMPLETE_MIN
            if not complete and not show_coarse:
                continue
            s = a[a.lr.round(8) == round(lr, 8)].sort_values("bpp")
            if s.empty:
                continue
            is_best = abs(lr - best) < 1e-9
            ax.plot(
                s.bpp,
                s.psnr,
                marker=LRMK[lr],
                color=col,
                ms=7,
                mfc=col if complete else "white",
                lw=3.0 if is_best else (1.8 if complete else 1.2),
                ls="-" if complete else "--",
                alpha=1.0 if complete else 0.75,
                zorder=6 if is_best else (4 if complete else 2),
            )
    ax.grid(alpha=0.3)
    ax.set_xlabel("bpp (bitstream)")
    ax.set_ylabel("PSNR [dB]")
    arch_h = [Line2D([], [], color=PAL[code], lw=3, label=lbl) for code, lbl, _ in ARCHS]
    lr_h = [
        Line2D([], [], color="0.3", marker=LRMK[lr], ls="none", ms=8, label=LRLBL[lr])
        for lr in LRS
    ]
    l1 = ax.legend(handles=arch_h, title="architecture", loc="lower right", fontsize=9)
    ax.add_artist(l1)
    ax.legend(
        handles=lr_h,
        title="learning rate",
        loc="lower right",
        bbox_to_anchor=(0.80, 0.0),
        fontsize=9,
    )
    ax.set_title(
        "lr-sweep overlay — RD curves (test_sub500, MERLIN ref)\n"
        "bold = production lr; "
        + ("solid=full / dashed=coarse sweep" if show_coarse else "full sweeps only"),
        fontsize=12,
    )
    return ax


for show_coarse in (False, True):
    fig, ax = plt.subplots(figsize=(9, 7))
    plot_lr_overlay(agg, nrun, show_coarse=show_coarse, ax=ax)
    fig.tight_layout()
    if SAVE_FIGURES:
        suffix = "coarse" if show_coarse else "clean"
        fig.savefig(PLOTS_DIR / f"lr_sweep_overlay_{suffix}.{FIGURE_FORMAT}", bbox_inches="tight")
        print("saved", PLOTS_DIR / f"lr_sweep_overlay_{suffix}.{FIGURE_FORMAT}")
    plt.show()